In [1]:
## init mongo db and fiftyone connection
import os

# Define the URI to point to your manual process
os.environ["FIFTYONE_DATABASE_URI"] = "mongodb://localhost:44123"

import fiftyone as fo

# Verify connection
print(fo.core.odm.database.get_db_conn()) 


You are running the oldest supported major version of MongoDB. Please refer to https://deprecation.voxel51.com for deprecation notices. You can suppress this exception by setting your `database_validation` config parameter to `False`. See https://docs.voxel51.com/user_guide/config.html#configuring-a-mongodb-connection for more information
Database(MongoClient(host=['localhost:44123'], document_class=dict, tz_aware=False, connect=True, appname='fiftyone'), 'fiftyone')


In [2]:
import fiftyone.brain as fob
from sklearn.preprocessing import normalize
import plotly.express as px
import skdim
import random
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import ot
from sklearn.manifold import TSNE
import cv2
from fiftyone import ViewField as F
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from pathlib import Path
import random
import glob 
import math
# ## renders plotly properly in a html instance. 
# import plotly.io as pio
# pio.renderers.default = "notebook"


In [3]:
## Load dataset and views from mongodb 
dataset = fo.load_dataset("dugong")

## load the views
nc_view = dataset.match(F('region').starts_with('NC'))
wp_view = dataset.match(F('region').starts_with('WP'))


In [9]:
dataset.save_view("nc", nc_view)
dataset.save_view("wp", wp_view)

In [11]:
dataset

Name:        dugong
Media type:  image
Num samples: 2755
Persistent:  True
Tags:        []
Sample fields:
    id:                    fiftyone.core.fields.ObjectIdField
    filepath:              fiftyone.core.fields.StringField
    tags:                  fiftyone.core.fields.ListField(fiftyone.core.fields.StringField)
    metadata:              fiftyone.core.fields.EmbeddedDocumentField(fiftyone.core.metadata.ImageMetadata)
    created_at:            fiftyone.core.fields.DateTimeField
    last_modified_at:      fiftyone.core.fields.DateTimeField
    region:                fiftyone.core.fields.StringField
    subregion:             fiftyone.core.fields.StringField
    mission_name:          fiftyone.core.fields.StringField
    sea_state:             fiftyone.core.fields.IntField
    turbidity_global:      fiftyone.core.fields.IntField
    turbidity_local:       fiftyone.core.fields.StringField
    sun_glitter:           fiftyone.core.fields.StringField
    cloud_reflection:      fiftyon

In [12]:

# create a combined key: e.g., "WP_UM_medium" or "WP_GAM_high"
# ensures the split respects both the location and the difficulty
dataset.set_values(
    "stratify_key",
    [f"{r}_{m}_{c}" for r, m, c in zip(
        dataset.values("region"), 
        dataset.values("subregion"), 
        dataset.values("mission_name")
    )]
)

print("Unique strata created:", dataset.count_values("stratify_key"))

Unique strata created: {'NC_NC_Flight_196': 63, 'NC_NC_Flight_233': 6, 'WP_FRIWEN_FRIWEN_M19': 14, 'NC_NC_Flight_209': 83, 'WP_MANTASANDY_MANTASANDY_M5': 3, 'NC_NC_Flight_219': 23, 'WP_FRIWEN_FRIWEN_M11': 335, 'NC_NC_Flight_199': 8, 'WP_UM_UM_M1': 14, 'NC_NC_Flight_211': 17, 'WP_UM_UM_M3': 4, 'NC_NC_Flight_212': 36, 'NC_NC_Flight_214': 19, 'NC_NC_Flight_205': 9, 'NC_NC_Flight_218': 22, 'NC_NC_Flight_198': 73, 'NC_NC_Flight_220': 20, 'NC_NC_Flight_225': 2, 'NC_NC_Flight_207': 128, 'WP_GAM_GAM_M2': 275, 'NC_NC_Flight_231': 12, 'NC_NC_Flight_235': 4, 'NC_NC_Flight_200': 10, 'NC_NC_Flight_215': 37, 'NC_NC_Flight_232': 23, 'WP_GAM_GAM_M10': 26, 'WP_UM_UM_M5': 449, 'WP_UM_UM_M2': 9, 'WP_UM_UM_M6': 269, 'NC_NC_Flight_195': 6, 'WP_GAM_GAM_M1': 211, 'NC_NC_Flight_213': 41, 'NC_NC_Flight_234': 2, 'WP_FRIWEN_FRIWEN_M12': 204, 'NC_NC_Flight_222': 17, 'WP_FRIWEN_FRIWEN_M8': 222, 'WP_FRIWEN_FRIWEN_M53': 4, 'NC_NC_Flight_197': 50, 'NC_NC_Flight_226': 5}


In [5]:
dataset.count_values('stratify_key')

{'WP_GAM_GAM_M1': 211,
 'NC_NC_Flight_198': 73,
 'NC_NC_Flight_234': 2,
 'NC_NC_Flight_207': 128,
 'WP_FRIWEN_FRIWEN_M12': 204,
 'NC_NC_Flight_219': 23,
 'WP_GAM_GAM_M2': 275,
 'NC_NC_Flight_197': 50,
 'WP_UM_UM_M3': 4,
 'NC_NC_Flight_213': 41,
 'NC_NC_Flight_222': 17,
 'NC_NC_Flight_231': 12,
 'WP_FRIWEN_FRIWEN_M8': 222,
 'WP_FRIWEN_FRIWEN_M11': 335,
 'WP_UM_UM_M1': 14,
 'NC_NC_Flight_196': 63,
 'NC_NC_Flight_200': 10,
 'WP_GAM_GAM_M10': 26,
 'NC_NC_Flight_199': 8,
 'NC_NC_Flight_212': 36,
 'WP_FRIWEN_FRIWEN_M53': 4,
 'NC_NC_Flight_233': 6,
 'NC_NC_Flight_211': 17,
 'WP_FRIWEN_FRIWEN_M19': 14,
 'WP_UM_UM_M6': 269,
 'NC_NC_Flight_232': 23,
 'NC_NC_Flight_218': 22,
 'NC_NC_Flight_195': 6,
 'NC_NC_Flight_226': 5,
 'WP_UM_UM_M2': 9,
 'WP_MANTASANDY_MANTASANDY_M5': 3,
 'NC_NC_Flight_235': 4,
 'NC_NC_Flight_209': 83,
 'WP_UM_UM_M5': 449,
 'NC_NC_Flight_225': 2,
 'NC_NC_Flight_205': 9,
 'NC_NC_Flight_215': 37,
 'NC_NC_Flight_214': 19,
 'NC_NC_Flight_220': 20}

In [17]:
wp_view.count_values('flight_plan')

{'FPLAN': 31, 'MAN': 2008}

### Random Stratified by Island
Random Stratified Sampling, which treats every image as an independent unit. This leads to every mission being "leaked" into every split.

In [6]:
from sklearn.model_selection import train_test_split
import pandas as pd


def tag_train_test_split_seeded(train_size:float,
                                test_size:float,
                                val_size:float,
                                runs:int,
                                dataset,
                                stratification_key = 'stratify_key',
                                ):
    """
    Creates the split for train, test and validation using a stratified pick given by the complexity. 
    Args:
        runs: Number of loop to pass and create the tag inside the dataset 
    
        Returns:
        Return the seed_number and the dataset get tagged.
    """
    # target islands (skipping MANTASANDY)
    islands_to_split = ['NC','UM', 'GAM', 'FRIWEN']
    seeds_list = []

    ## tags adding train_seed_number for west papua 
    for run in range(0,runs+1):
        seed_number  = random.randint(1,50)
        print(f"Seed number selected:{seed_number}")
        seeds_list.append(seed_number)
        for island in islands_to_split:
            # Get a view of just this island
            island_view = dataset.match(F("subregion") == island)
            
            ids = island_view.values("id")

            # stratification field presented in the dataset 
            strata = island_view.values(stratification_key)
            
            # Split off the TEST set (20%) ---
            # Stratify ensures the 'high complexity' ratio stays the same
            train_val_ids, test_ids = train_test_split(
                ids, 
                test_size= test_size, 
                stratify=strata,
                shuffle=True, 
                random_state=seed_number
            )
            
            # Get strata for the remaining 80% to split again
            train_val_strata = [s for i, s in zip(ids, strata) if i in train_val_ids]
            
            # Split remaining 80% into Train (70% total) and Val (10% total) ---
            # 0.125 * 0.8 = 0.1 (which is 10% of the original total)
            train_ids, val_ids = train_test_split(
                train_val_ids, 
                test_size= (val_size/(1-test_size)), 
                stratify=train_val_strata, 
                random_state=seed_number
            )
            
            # 3. Apply the tags in FiftyOne
            dataset.select(train_ids).tag_samples(f"train_{str(seed_number)}")
            dataset.select(val_ids).tag_samples(f"val_{str(seed_number)}")
            dataset.select(test_ids).tag_samples(f"test_{str(seed_number)}")
            
            print(f"Island {island}: Train={len(train_ids)}, Test={len(test_ids)}, Val={len(val_ids)}")

    return seeds_list


In [22]:
# This ensures we pull and push values in the exact same sample order
vals = dataset.values("stratify_key")
new_vals = ["NC_only" if v.split('_')[0] == 'NC' else v for v in vals]

dataset.set_values("new_stratify_key", new_vals)

In [12]:
## use the function to run 
folder =  '/share/home/e2406743/dataset/exported_img/seed_42'
## percentage of each train, val, test
train_size = 0.7
test_size = 0.15 
val_size = 0.15
runs = 1

seed_number_list = tag_train_test_split_seeded(train_size, test_size, val_size,
                                          runs=runs,
                                          dataset= dataset,
                                          stratification_key='stratify_key'
                                          )

## tag TRAIN for all new caledonia samples.
#nc_view = dataset.match(F("region") == "NC")
#nc_view.tag_samples("train")

Seed number selected:11
Island NC: Train=500, Test=108, Val=108
Island UM: Train=521, Test=112, Val=112
Island GAM: Train=358, Test=77, Val=77
Island FRIWEN: Train=545, Test=117, Val=117
Seed number selected:37
Island NC: Train=500, Test=108, Val=108
Island UM: Train=521, Test=112, Val=112
Island GAM: Train=358, Test=77, Val=77
Island FRIWEN: Train=545, Test=117, Val=117


In [30]:
dataset.distinct('tags')

['test_11', 'test_37', 'train_11', 'train_37', 'val_11', 'val_37']

In [13]:
def get_mission_split_report(dataset, seed_number, mission_field="mission_name"):
    """
    Creates a summary of which flight missions are in which split for a given seed.
    """
    tags = [f"train_{seed_number}", f"val_{seed_number}", f"test_{seed_number}"]
    report_data = []

    # Get all unique missions
    all_missions = dataset.distinct(mission_field)

    for mission in all_missions:
        mission_view = dataset.match(F(mission_field) == mission)
        
        # Count samples per tag for this specific mission
        counts = {
            "mission": mission,
            "total": len(mission_view),
            "train": len(mission_view.match_tags(tags[0])),
            "val":   len(mission_view.match_tags(tags[1])),
            "test":  len(mission_view.match_tags(tags[2]))
        }
        report_data.append(counts)

    df = pd.DataFrame(report_data)
    # Add a percentage column to see if a mission is "leaking" or dominated by one split
    df['train_pct'] = (df['train'] / df['total'] * 100).round(1)
    
    return df

# Example usage for your first seed:
first_seed = seed_number_list[0]
mission_df = get_mission_split_report(dataset, first_seed)
print(mission_df)

          mission  total  train  val  test  train_pct
0      FRIWEN_M11    335    238   47    50       71.0
1      FRIWEN_M12    204    145   28    31       71.1
2      FRIWEN_M19     14      9    3     2       64.3
3      FRIWEN_M53      4      3    0     1       75.0
4       FRIWEN_M8    222    150   39    33       67.6
5      Flight_195      6      4    1     1       66.7
6      Flight_196     63     45    9     9       71.4
7      Flight_197     50     36    6     8       72.0
8      Flight_198     73     49   13    11       67.1
9      Flight_199      8      4    3     1       50.0
10     Flight_200     10      6    2     2       60.0
11     Flight_205      9      5    3     1       55.6
12     Flight_207    128     93   16    19       72.7
13     Flight_209     83     62    8    13       74.7
14     Flight_211     17     12    2     3       70.6
15     Flight_212     36     25    6     5       69.4
16     Flight_213     41     29    6     6       70.7
17     Flight_214     19    

In [15]:
mission_df[['train','val','test']].sum(axis=0)

train    1924
val       414
test      414
dtype: int64

## Grouped Stratified Splitting
Every mission flight is the atomic unit that stays together

In [11]:
out_dict = []
for island in ['FRIWEN','GAM','UM','MANTASANDY']:
    vieww = dataset.match(F("subregion")==island)

    missions = vieww.distinct("mission_name")
    mission_counts = {m: len(vieww.match(F("mission_name") == m)) for m in missions}
    out_dict.append(mission_counts)

In [12]:
dict_out = {}
for dic in out_dict:
    for k,v in dic.items():
        dict_out[k] = v

dict_out

{'FRIWEN_M11': 335,
 'FRIWEN_M12': 204,
 'FRIWEN_M19': 14,
 'FRIWEN_M53': 4,
 'FRIWEN_M8': 222,
 'GAM_M1': 211,
 'GAM_M10': 26,
 'GAM_M2': 275,
 'UM_M1': 14,
 'UM_M2': 9,
 'UM_M3': 4,
 'UM_M5': 449,
 'UM_M6': 269,
 'MANTASANDY_M5': 3}

In [ ]:
first_seed = seed_number_list[0]
mission_df = get_mission_split_report(dataset, first_seed)
print(mission_df)

### delete old tags

In [10]:
distinct_old_tags = dataset.distinct("tags")
distinct_old_tags

['test_11', 'test_19', 'train_11', 'train_19', 'val_11', 'val_19']

In [11]:
dataset.untag_samples(distinct_old_tags)

# split - train, test, val for full images paths 
## create csv filepaths:

In [18]:
def return_list_filepath_train_test_val(seed_number, nc_view, wp_view):
    train_seed_filepath = wp_view.match_tags(f"train_{seed_number}").values("filepath")
    test_seed_filepath = wp_view.match_tags(f"test_{seed_number}").values("filepath")
    val_seed_filepath = wp_view.match_tags(f"val_{seed_number}").values("filepath")
    ## train_nc_filepath = nc_view.match_tags("train").values("filepath") OLD
    train_nc_filepath = nc_view.match_tags(f"train_{seed_number}").values("filepath")
    return train_seed_filepath, test_seed_filepath, val_seed_filepath, train_nc_filepath


def build_filepath_df(train_seed_filepath, test_seed_filepath, val_seed_filepath, train_nc_filepath):

    df = pd.DataFrame({
        "train_seed": pd.Series(train_seed_filepath),
        "test_seed": pd.Series(test_seed_filepath),
        "val_seed": pd.Series(val_seed_filepath),
        "train_nc": pd.Series(train_nc_filepath),
    })

    return df



## IMPLEMENT LOOP HERE
## RUN ALL GIVEN SEEDS AND CREATES A CSV WITH THE PATHS REGARDING THE FULL IMAGE
for ss in seed_number_list:
    print(f"Running seed:{ss}")
    (train_seed_filepath, test_seed_filepath,
    val_seed_filepath, train_nc_filepath) = return_list_filepath_train_test_val(
        ss,
        nc_view=nc_view,
        wp_view = wp_view
    )

    df_seed = build_filepath_df(
        train_seed_filepath,
        test_seed_filepath,
        val_seed_filepath,
        train_nc_filepath
    )

    ## save it keeping 
    output_filename = f"df_train_test_split_filepath_{str(ss)}.csv"
    print(f"saving file:{output_filename}")
    output_folder = "/share/home/e2406743/dataset/df_filepaths"
    os.makedirs(output_folder, exist_ok=True)
    print(f"saving at:{os.path.join(output_folder,output_filename)}")
    df_seed.to_csv(os.path.join(output_folder, output_filename))
    print('done!')


Running seed:11
saving file:df_train_test_split_filepath_11.csv
saving at:/share/home/e2406743/dataset/df_filepaths/df_train_test_split_filepath_11.csv
done!
Running seed:37
saving file:df_train_test_split_filepath_37.csv
saving at:/share/home/e2406743/dataset/df_filepaths/df_train_test_split_filepath_37.csv
done!


## load back

In [19]:
def get_seed_from_filepath(csv_file):
    path = Path(csv_file).stem
    return path.split('_')[-1]

def return_list_from_csv(csv_file):
    dff = pd.read_csv(csv_file)
    wp_train_list = dff['train_seed'].dropna().values
    test_list = dff['test_seed'].dropna().values
    val_list = dff['val_seed'].dropna().values
    nc_train_list = dff['train_nc'].dropna().values
    return wp_train_list, nc_train_list, test_list, val_list

In [21]:
csv_file  = '/share/home/e2406743/dataset/df_filepaths/df_train_test_split_filepath_37.csv'
get_seed_from_filepath(csv_file)

'37'

### drafts

In [11]:
from sklearn.model_selection import train_test_split

In [ ]:
##csv_file='/share/home/e2406743/dataset/df_filepaths/df_train_test_split_filepath_38.csv'
wp_train_list, nc_train_list, test_list, val_list = return_list_from_csv(csv_file)

ids = np.array(ids)
train_size = 0.7
test_size = 0.2
val_size = 0.1
seed_number = int(get_seed_from_filepath(csv_file))
assert type(seed_number)==int

# Split off the TEST set (20%) ---
# Stratify ensures the 'high complexity' ratio stays the same
train_val_ids, test_ids = train_test_split(
    ids, 
    test_size= test_size, 
    shuffle=True, 
    random_state=seed_number
)


# Split remaining 80% into Train (70% total) and Val (10% total) ---
# 0.125 * 0.8 = 0.1 (which is 10% of the original total)
train_ids, val_ids = train_test_split(
    train_val_ids, 
    test_size= (val_size/(1-test_size)),
    random_state=seed_number
)

In [18]:
def create_train_test_split_for_nc(nc_train_list):
    from sklearn.model_selection import train_test_split
    train_size = 0.7
    test_size = 0.2
    val_size = 0.1
    seed_number = int(get_seed_from_filepath(csv_file))
    assert type(seed_number)==int

    # Split off the TEST set (20%) ---
    # Stratify ensures the 'high complexity' ratio stays the same
    train_val_ids, test_ids = train_test_split(
        ids, 
        test_size= test_size, 
        shuffle=True, 
        random_state=seed_number
    )


    # Split remaining 80% into Train (70% total) and Val (10% total) ---
    # 0.125 * 0.8 = 0.1 (which is 10% of the original total)
    train_ids, val_ids = train_test_split(
        train_val_ids, 
        test_size= (val_size/(1-test_size)),
        random_state=seed_number
    )
    return train_ids, test_ids, val_ids
    

In [19]:
train_list_ncnc, test_list_ncnc, val_list_ncnc = create_train_test_split_for_nc(nc_train_list)

In [20]:
print(len(train_list_ncnc), len(test_list_ncnc), len(val_list_ncnc))

500 144 72


In [ ]:
from sklearn.model_selection import train_test_split

strata = nc_view.values("background_complexity")
            
# Split off the TEST set (20%) ---
# Stratify ensures the 'high complexity' ratio stays the same
train_val_ids, test_ids = train_test_split(
    ids, 
    test_size= test_size, 
    stratify=strata,
    shuffle=True, 
    random_state=seed_number
)

'/share/home/e2406743/dataset/dataset/NC/Flight_226/images/GH034226-619fa2d56d4d3_92.jpeg'

here we have a list containing the filepath of the full images. From these lists, I can randomly select different percentage regarding the whole list size and also using fiftyone with uniqueness. 
After this, it is necessary to set up the mapping dict to map the full images with the associated tiles.

In [13]:
len(wp_train_list)

1423

In [28]:

def random_choice_train_list(train_list,seed,partitions:list=[0.1,0.25,0.5,0.75,1.0]):
    length = len(train_list)

    ## seed 
    random.seed(seed)

    dict_out ={}
    for p in partitions:
        num_images = int(math.floor(length*p))
        dict_out[f"partition_{str(int(p*100))}"] = random.choices(train_list, k=num_images)
    
    return dict_out

In [34]:
## seed with the same seed used to create the partition.
my_seed = random.seed(int(get_seed_from_filepath(csv_file)))

dictt = random_choice_train_list(train_list = wp_train_list,
                                    seed = my_seed,
                                    partitions = [0.05,0.1,0.25,0.5,0.75,1.0]
                                )

NameError: name 'random_choice_train_list' is not defined

In [16]:
len(dictt['partition_25']), len(dictt['partition_75'])

(355, 1067)

In [27]:
def get_files_by_stem(filepath_stem, patch_folder):
    dict_out = {}
    foolder_meta = os.path.join(patch_folder, 'metadata')
    list_meta = list(glob.glob(os.path.join(foolder_meta, f'{filepath_stem}__*.json')))
    foolder_meta = os.path.join(patch_folder, 'images')
    list_images = list(glob.glob(os.path.join(foolder_meta, f'{filepath_stem}__*.jpg')))
    foolder_meta = os.path.join(patch_folder, 'labels')
    list_labels = list(glob.glob(os.path.join(foolder_meta, f'{filepath_stem}__*.txt')))
    dict_out['metadata'] = list_meta
    dict_out['label'] = list_labels
    dict_out['images'] = list_images
    return dict_out

## use the map dict to create the final list of filepaths regarding the patches 
def mapdict_patches_filepath(list_paths, patch_folder):
    dict_map_filepath = {}
    for path in list_paths:
        stem = Path(path).stem
        dict_map_filepath[stem] = get_files_by_stem(stem, patch_folder)

    ## flat dict
    filepath_all_images   = [f for d in dict_map_filepath.values() for f in d.get('images', [])]
    filepath_all_labels   = [f for d in dict_map_filepath.values() for f in d.get('label', [])]
    filepath_all_metadata = [f for d in dict_map_filepath.values() for f in d.get('metadata', [])]

    ## -------------
    print(f'images:{len(filepath_all_images)}')
    print(f"labels:{len(filepath_all_labels)}")
    print(f"metadata:{len(filepath_all_metadata)}")

    return filepath_all_images, filepath_all_labels, filepath_all_metadata

In [18]:
list_images, list_labels, list_metadata = mapdict_patches_filepath(dictt['partition_25'])

images:1089
labels:1089
metadata:1089


# full pipeline

In [22]:
import re 
[match for x in dataset.distinct("tags") for match in re.findall(r"\d+",x)]

['11', '37', '11', '37', '11', '37']

In [23]:
dataset.distinct("tags")


['test_11', 'test_37', 'train_11', 'train_37', 'val_11', 'val_37']

In [24]:
def get_seed_number_from_tags(tags_list):
    import re 
    return pd.Series([int(match) for x in tags_list for match in re.findall(r"\d+",x)]).unique()


In [25]:
get_seed_number_from_tags(tags_list = dataset.distinct("tags"))

array([11, 37])

In [29]:
## use the function to run 
csv_file='/share/home/e2406743/dataset/df_filepaths/df_train_test_split_filepath_37.csv'
output_folder = "/share/home/e2406743/dataset/df_filepaths"
patch_folder = "/share/home/e2406743/dataset/exported_img/seed_42"

## this control if it will split and tag the dataset. However, there is no necessity to do it again once its already done.
## so turning false skips this section and look the folder with the already existent csv_filepaths of the FULL IMAGE
split_and_tag  = False
## percentage of each train, val, test
train_size = 0.7
test_size = 0.15 
val_size = 0.15

runs = 1 # number of seeds to create, if 1 is equals to:2. The loop always sums 1 +=

if split_and_tag:
    ## create TAGS in the dataset. train_seednumber, test, val
    seed_number_list = tag_train_test_split_seeded(train_size, test_size, val_size,
                                            runs=runs,
                                            dataset= dataset,
                                            stratification_key='stratify_key'
                                            )
    
    ## NC TRAIN IS BEING SAMPLED INSIDE THE TAG FUNCTION
    # ## tag TRAIN for all new caledonia samples.
    # nc_view = dataset.match(F("region") == "NC")
    # nc_view.tag_samples("train")

    ## get all the unique seeds within the TAGS
    seed_number_list = get_seed_number_from_tags(tags_list = dataset.distinct("tags"))

    ## filter the tag
    ## RUN ALL GIVEN SEEDS AND CREATES A CSV WITH THE PATHS REGARDING THE FULL IMAGE
    for ss in seed_number_list:
        print(f"Running seed:{ss}")
        train_seed_filepath, test_seed_filepath, val_seed_filepath, train_nc_filepath = return_list_filepath_train_test_val(
            ss, dataset=dataset, nc_view=nc_view
        )

        df_seed = build_filepath_df(
            train_seed_filepath,
            test_seed_filepath,
            val_seed_filepath,
            train_nc_filepath
        )

        ## save it keeping 
        output_filename = f"df_train_test_split_filepath_{str(ss)}.csv"
        print(f"saving file:{output_filename}")
        os.makedirs(output_folder, exist_ok=True)
        print(f"saving at:{os.path.join(output_folder,output_filename)}")
        df_seed.to_csv(os.path.join(output_folder, output_filename))
        print('done!')


## --------------------------------------------------
## HERE IT IS BEING DOING FOR ONE SEED THAT WAS TAGGED IN TAGS
#### LOAD CSV FILEPATH BACK 

wp_train_list, nc_train_list, test_list, val_list = return_list_from_csv(csv_file)

## seed with the same seed used to create the partition.
my_seed = int(get_seed_from_filepath(csv_file))
print(f"seeding :{my_seed}")
random.seed(my_seed)

## TRAIN WP--------------------
## create a dict containing the filepath_stem with the keys containig the filepath for images, labels and metadata
dictt = random_choice_train_list(train_list = wp_train_list,
                                    seed = my_seed,
                                    partitions = [0.05,0.1,0.25,0.5,0.75,1.0]
                                )


## BEST SET 
## make a functoin to choose the best set of images 
## TODO

## LOOP into each dictt key and return a full dataset
## return for each partition the paths associated for images, labels and metadata.

new_dict = dictt.copy()
output_dict_partitions = dict()

## each key is a full filepath
for key in new_dict.keys():
    print(f"Running key:{key}")

    ## retriveves for each partition the associated patches, labels, metadata  - filepath 
    list_images, list_labels, list_metadata = mapdict_patches_filepath(dictt[key], patch_folder)
    output_dict_partitions[key] = {'images':list_images , 'labels':list_labels, 'metadata': list_metadata}

## TRAIN WP- SAVE DF 
print("\n")
print("saving patches filepath with the partition and selected by the given seed")
df_patches_filepath = pd.DataFrame().from_dict(output_dict_partitions)
output_filename = f"df_train_test_split_filepath_PATCHES_wpartitions_seed_{str(my_seed)}.parquet"
print(f"saving file:{output_filename}")
os.makedirs(output_folder, exist_ok=True)
print(f"saving at:{os.path.join(output_folder,output_filename)}")
df_patches_filepath.to_parquet((os.path.join(output_folder, output_filename)))



seeding :37
Running key:partition_5


images:234
labels:234
metadata:234
Running key:partition_10
images:444
labels:444
metadata:444
Running key:partition_25
images:1044
labels:1044
metadata:1044
Running key:partition_50
images:1961
labels:1961
metadata:1961
Running key:partition_75
images:2625
labels:2625
metadata:2625
Running key:partition_100
images:3008
labels:3008
metadata:3008


saving patches filepath with the partition and selected by the given seed
saving file:df_train_test_split_filepath_PATCHES_wpartitions_seed_37.parquet
saving at:/share/home/e2406743/dataset/df_filepaths/df_train_test_split_filepath_PATCHES_wpartitions_seed_37.parquet
